# Revisão reproduzível do treinamento — 24/08/2026

## tl;dr

Notebook somente-leitura para reexecutar a auditoria de status, métricas de teste, previsões e telemetria dos runs do benchmark. Executar a partir da raiz do repositório.

## Context & Methods

A unidade de análise é um run por dataset e configuração. Os resultados são lidos diretamente de `status.json`, `epoch_metrics.csv`, `test_metrics.json`, `predictions.csv` e `telemetry/samples.csv`. Nenhum arquivo do treinamento é alterado.

In [ ]:
from pathlib import Path
import csv
import json
from collections import Counter

ROOT = Path('outputs/controlled-augmentation2-mac-m4')
runs = []
for status_path in ROOT.glob('batch/**/runs/*/status.json'):
    with status_path.open() as f:
        status = json.load(f)
    if status.get('status') not in {'completed', 'running'}:
        continue
    run = status_path.parent
    parts = run.parts
    i = parts.index('batch')
    runs.append({'batch': parts[i + 1], 'dataset': parts[i + 2], 'status': status['status'], 'root': run})
print(Counter((r['status'] for r in runs)))


## Data

### Métricas por run


In [ ]:
for item in sorted(runs, key=lambda x: (x['dataset'], x['batch'])):
    metrics_path = item['root'] / 'checkpoints/epoch_metrics.csv'
    if not metrics_path.exists():
        continue
    with metrics_path.open(newline='') as f:
        history = list(csv.DictReader(f))
    best = max(history, key=lambda r: float(r['val_macro_f1']))
    last = history[-1]
    result = {'run': f"{item['dataset']} {item['batch']}", 'status': item['status'], 'epochs': len(history), 'best_val_macro_f1': float(best['val_macro_f1']), 'last_val_macro_f1': float(last['val_macro_f1'])}
    test_path = item['root'] / 'artifacts/test_metrics.json'
    if test_path.exists():
        with test_path.open() as f:
            classification = json.load(f)['classification']
        result.update(test_macro_f1=classification['macro_f1'], test_accuracy=classification['accuracy'])
    print(result)


### Previsões e telemetria


In [ ]:
for item in sorted(runs, key=lambda x: (x['dataset'], x['batch'])):
    pred_path = item['root'] / 'artifacts/predictions.csv'
    tele_path = item['root'] / 'telemetry/samples.csv'
    if pred_path.exists():
        with pred_path.open(newline='') as f:
            predictions = list(csv.DictReader(f))
        errors = [r for r in predictions if r['true_label'] != r['predicted_label']]
        top_pairs = Counter((r['true_label'], r['predicted_label']) for r in errors).most_common(3)
        print(f"{item['dataset']} {item['batch']}: predictions={len(predictions)}, errors={len(errors)}, top_pairs={top_pairs}")
    if tele_path.exists():
        with tele_path.open(newline='') as f:
            telemetry = list(csv.DictReader(f))
        print(f"  telemetry_rows={len(telemetry)}, fields={len(telemetry[0]) if telemetry else 0}, last={telemetry[-1]['timestamp'] if telemetry else None}")


## Results

Os resultados esperados são cinco runs completos com teste e previsões, um run ativo sem avaliação final e os demais jobs pendentes.

## Takeaways

- O melhor MNIST concluído é o batch 64.
- Fashion-MNIST apresenta desempenho menor e confusões concentradas entre classes visualmente semelhantes.
- O consolidado final só deve ser produzido após concluir batch, quantização e ativação.